In [ ]:
#Project setup

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: d:\AI Projects\Credit-Risk-Knowledge-Assistant


In [ ]:
#Open existing ChromaDB

from app.services.vector_store import get_vector_store

In [ ]:
#Open existing ChromaDB

vector_store = get_vector_store()

print("Vector store loaded successfully.")

d:\AI Projects\Credit-Risk-Knowledge-Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4287.21it/s]


Vector store loaded successfully.


In [4]:
## Baseline Similarity Retriever

similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [5]:
# Test the retriever

question = "What is the objective of Ind AS 109?"

results = similarity_retriever.invoke(question)

print("Retrieved documents:", len(results))

Retrieved documents: 5


In [6]:
for rank, doc in enumerate(results, start=1):

    print("=" * 80)

    print("Rank:", rank)

    print(
        "Source:",
        doc.metadata.get("source_file")
    )

    print(
        "Page:",
        doc.metadata.get("page_number")
    )

    print(
        "Chunk ID:",
        doc.metadata.get("chunk_id")
    )

    print()

    print(doc.page_content[:700])

    print()

Rank: 1
Source: INDAS109.pdf
Page: 188
Chunk ID: 621

433 
 
Appendix E 
References to matters contained in other Indian 
Accounting Standards 
This appendix is an integral part of the Ind AS. 
 
This appendix lists the appendices which are part of other Indian Accounting 
Standards and make reference to Ind AS 109,Financial Instruments. 
1. Appendix A,Rights to Interests arising from Decommissioning,Restoration 
and Environmental Rehabilitation contained in Ind AS 37 , Provisions, 
Contingent Liabilities and Contingent Assets. 
 
2. Appendix C, Service Concession Arrangements  contained in Ind AS 115, 
Revenue from Contracts with Customers. 
 
3. Appendix B, Evaluating the Substance of Transactions Involving the Legal 
Form of Lease contained 

Rank: 2
Source: INDAS109.pdf
Page: 173
Chunk ID: 580

418 
 
currency risks that qualify as a hedged risk in the hedge of a net investment in a 
foreign operation. 
 
5 Ind AS 109 allows an entity to designate either a de rivative or a non -der

In [7]:
# Test different k values

for k in [2, 3, 5, 8]:

    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": k
        }
    )

    docs = retriever.invoke(question)

    print("\n" + "=" * 80)
    print(f"k = {k}")

    for rank, doc in enumerate(docs, start=1):

        print(
            rank,
            "|",
            doc.metadata.get("source_file"),
            "| Page:",
            doc.metadata.get("page_number")
        )


k = 2
1 | INDAS109.pdf | Page: 188
2 | INDAS109.pdf | Page: 173

k = 3
1 | INDAS109.pdf | Page: 188
2 | INDAS109.pdf | Page: 173
3 | INDAS109.pdf | Page: 172

k = 5
1 | INDAS109.pdf | Page: 188
2 | INDAS109.pdf | Page: 173
3 | INDAS109.pdf | Page: 172
4 | INDAS109.pdf | Page: 31
5 | Master Circular - Prudential norms on Income Recognition, Asset Classification and.pdf | Page: 52

k = 8
1 | INDAS109.pdf | Page: 188
2 | INDAS109.pdf | Page: 173
3 | INDAS109.pdf | Page: 172
4 | INDAS109.pdf | Page: 31
5 | Master Circular - Prudential norms on Income Recognition, Asset Classification and.pdf | Page: 52
6 | INDAS109.pdf | Page: 173
7 | Master Circular - Prudential norms on Income Recognition, Asset Classification and.pdf | Page: 2
8 | INDAS109.pdf | Page: 1


In [8]:
## MMR Retrieval

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

In [9]:
mmr_results = mmr_retriever.invoke(question)

In [10]:
for rank, doc in enumerate(
    mmr_results,
    start=1
):

    print("=" * 80)

    print("Rank:", rank)

    print(
        "Source:",
        doc.metadata.get("source_file")
    )

    print(
        "Page:",
        doc.metadata.get("page_number")
    )

    print()

    print(doc.page_content[:600])

Rank: 1
Source: INDAS109.pdf
Page: 188

433 
 
Appendix E 
References to matters contained in other Indian 
Accounting Standards 
This appendix is an integral part of the Ind AS. 
 
This appendix lists the appendices which are part of other Indian Accounting 
Standards and make reference to Ind AS 109,Financial Instruments. 
1. Appendix A,Rights to Interests arising from Decommissioning,Restoration 
and Environmental Rehabilitation contained in Ind AS 37 , Provisions, 
Contingent Liabilities and Contingent Assets. 
 
2. Appendix C, Service Concession Arrangements  contained in Ind AS 115, 
Revenue from Contracts with Customers. 
 
3
Rank: 2
Source: INDAS109.pdf
Page: 31

contingent consideration recognised by an acquirer in a business 
combination to which Ind AS 103 applies. (See paragrap h B5.7.3 
for guidance on foreign exchange gains or losses.)  
 
5.7.6 If an entity makes the election in paragraph 5.7.5, it shall recognise in 
profit or loss dividends from that investment in acco

What is MMR?
MMR means:
Maximal Marginal Relevance

In [11]:
# Compare Similarity vs MMR


def display_results(title, documents):

    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)

    for rank, doc in enumerate(
        documents,
        start=1
    ):

        print(
            f"\nRank {rank}"
        )

        print(
            "Source:",
            doc.metadata.get("source_file")
        )

        print(
            "Page:",
            doc.metadata.get("page_number")
        )

        print(
            "Chunk:",
            doc.metadata.get("chunk_id")
        )

        print()

        print(
            doc.page_content[:500]
        )

In [12]:
similarity_results = (
    similarity_retriever.invoke(question)
)

mmr_results = (
    mmr_retriever.invoke(question)
)

In [13]:
display_results(
    "SIMILARITY",
    similarity_results
)

display_results(
    "MMR",
    mmr_results
)


SIMILARITY

Rank 1
Source: INDAS109.pdf
Page: 188
Chunk: 621

433 
 
Appendix E 
References to matters contained in other Indian 
Accounting Standards 
This appendix is an integral part of the Ind AS. 
 
This appendix lists the appendices which are part of other Indian Accounting 
Standards and make reference to Ind AS 109,Financial Instruments. 
1. Appendix A,Rights to Interests arising from Decommissioning,Restoration 
and Environmental Rehabilitation contained in Ind AS 37 , Provisions, 
Contingent Liabilities and Contingent Assets. 
 
2. Appendix C, Se

Rank 2
Source: INDAS109.pdf
Page: 173
Chunk: 580

418 
 
currency risks that qualify as a hedged risk in the hedge of a net investment in a 
foreign operation. 
 
5 Ind AS 109 allows an entity to designate either a de rivative or a non -derivative 
financial instrument (or a combination of derivative and non -derivative financial 
instruments) as hedging instruments for foreign currency risk. This Appendix 
provides guidance on whe

In [14]:
## Similarity Threshold Retriever

threshold_retriever = (
    vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": 5,
            "score_threshold": 0.4
        }
    )
)

In [15]:
threshold_results = (
    threshold_retriever.invoke(question)
)

print(
    "Documents returned:",
    len(threshold_results)
)

Documents returned: 5


In [16]:
# Test an unrelated question

unrelated_question = (
    "How can I cook Italian pasta?"
)

In [17]:
docs = threshold_retriever.invoke(
    unrelated_question
)

print(
    "Retrieved:",
    len(docs)
)

No relevant docs were retrieved using the relevance score threshold 0.4


Retrieved: 0


In [18]:
## Metadata Filtering

indas_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {
            "source_file": "INDAS109.pdf"
        }
    }
)

In [19]:
results = indas_retriever.invoke(
    "What is expected credit loss?"
)

In [20]:
for doc in results:

    print(
        doc.metadata.get("source_file")
    )

INDAS109.pdf
INDAS109.pdf
INDAS109.pdf
INDAS109.pdf
INDAS109.pdf


##### Build a small retrieval evaluation set

In [21]:
test_questions = [
    "What is the objective of Ind AS 109?",
    "What is expected credit loss?",
    "What is significant increase in credit risk?",
    "When should lifetime expected credit losses be recognised?",
    "What are the rules for income recognition on NPAs?",
    "What are the prudential norms for asset classification?",
    "What guidelines apply to bank finance to NBFCs?"
]

In [22]:
for question in test_questions:

    print("\n" + "#" * 100)

    print("QUESTION:")
    print(question)

    results = similarity_retriever.invoke(
        question
    )

    for rank, doc in enumerate(
        results[:3],
        start=1
    ):

        print(
            f"\nRank {rank}:"
        )

        print(
            doc.metadata.get(
                "source_file"
            )
        )

        print(
            "Page:",
            doc.metadata.get(
                "page_number"
            )
        )

        print(
            doc.page_content[:350]
        )


####################################################################################################
QUESTION:
What is the objective of Ind AS 109?

Rank 1:
INDAS109.pdf
Page: 188
433 
 
Appendix E 
References to matters contained in other Indian 
Accounting Standards 
This appendix is an integral part of the Ind AS. 
 
This appendix lists the appendices which are part of other Indian Accounting 
Standards and make reference to Ind AS 109,Financial Instruments. 
1. Appendix A,Rights to Interests arising from Decommissioning,

Rank 2:
INDAS109.pdf
Page: 173
418 
 
currency risks that qualify as a hedged risk in the hedge of a net investment in a 
foreign operation. 
 
5 Ind AS 109 allows an entity to designate either a de rivative or a non -derivative 
financial instrument (or a combination of derivative and non -derivative financial 
instruments) as hedging instruments for foreign currency risk. This

Rank 3:
INDAS109.pdf
Page: 172
417 
 
Appendix C 
Hedges of a Net Investment in a Fo

In [23]:
evaluation_results = []

In [24]:
{
    "question": "...",
    "expected_source": "...",
    "expected_page": 1,
    "retrieved_pages": [...],
    "hit_at_1": True,
    "hit_at_3": True,
    "hit_at_5": True
}

{'question': '...',
 'expected_source': '...',
 'expected_page': 1,
 'retrieved_pages': [Ellipsis],
 'hit_at_1': True,
 'hit_at_3': True,
 'hit_at_5': True}

In [25]:
evaluation_set = [
    {
        "question":
            "What is the objective of Ind AS 109?",

        "expected_source":
            "INDAS109.pdf",

        "expected_page":
            1
    }
]

In [28]:
##Calculate Hit@K

def evaluate_hit_at_k(
    retriever,
    question,
    expected_source,
    expected_page,
    k=5
):

    documents = retriever.invoke(
        question
    )

    documents = documents[:k]

    for rank, document in enumerate(
        documents,
        start=1
    ):

        source = document.metadata.get(
            "source_file"
        )

        page = document.metadata.get(
            "page_number"
        )

        if (
            source == expected_source
            and page == expected_page
        ):

            return {
                "hit": 1,
                "rank": rank,
                "reciprocal_rank":
                    1 / rank
            }

    return {
        "hit": 0,
        "rank": None,
        "reciprocal_rank": 0
    }

In [29]:
result = evaluate_hit_at_k(
    retriever=similarity_retriever,
    question="What is the objective of Ind AS 109?",
    expected_source="INDAS109.pdf",
    expected_page=1,
    k=5
)

result

{'hit': 0, 'rank': None, 'reciprocal_rank': 0}

In [30]:
question = "What is the objective of Ind AS 109?"

docs = similarity_retriever.invoke(question)

for rank, doc in enumerate(docs, start=1):
    print("=" * 80)
    print("Rank:", rank)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page_number"))
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print()
    print(doc.page_content[:500])

Rank: 1
Source: INDAS109.pdf
Page: 188
Chunk ID: 621

433 
 
Appendix E 
References to matters contained in other Indian 
Accounting Standards 
This appendix is an integral part of the Ind AS. 
 
This appendix lists the appendices which are part of other Indian Accounting 
Standards and make reference to Ind AS 109,Financial Instruments. 
1. Appendix A,Rights to Interests arising from Decommissioning,Restoration 
and Environmental Rehabilitation contained in Ind AS 37 , Provisions, 
Contingent Liabilities and Contingent Assets. 
 
2. Appendix C, Se
Rank: 2
Source: INDAS109.pdf
Page: 173
Chunk ID: 580

418 
 
currency risks that qualify as a hedged risk in the hedge of a net investment in a 
foreign operation. 
 
5 Ind AS 109 allows an entity to designate either a de rivative or a non -derivative 
financial instrument (or a combination of derivative and non -derivative financial 
instruments) as hedging instruments for foreign currency risk. This Appendix 
provides guidance on where, wi

In [31]:
results = vector_store.similarity_search_with_score(
    query="What is the objective of Ind AS 109?",
    k=20
)

for rank, (doc, score) in enumerate(results, start=1):

    print("=" * 80)
    print("Rank:", rank)
    print("Distance:", round(score, 4))
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page_number"))

    print()
    print(doc.page_content[:300])

Rank: 1
Distance: 0.4775
Source: INDAS109.pdf
Page: 188

433 
 
Appendix E 
References to matters contained in other Indian 
Accounting Standards 
This appendix is an integral part of the Ind AS. 
 
This appendix lists the appendices which are part of other Indian Accounting 
Standards and make reference to Ind AS 109,Financial Instruments. 
1. Appendix A
Rank: 2
Distance: 0.494
Source: INDAS109.pdf
Page: 173

418 
 
currency risks that qualify as a hedged risk in the hedge of a net investment in a 
foreign operation. 
 
5 Ind AS 109 allows an entity to designate either a de rivative or a non -derivative 
financial instrument (or a combination of derivative and non -derivative financial 
instruments) as h
Rank: 3
Distance: 0.5171
Source: INDAS109.pdf
Page: 172

417 
 
Appendix C 
Hedges of a Net Investment in a Foreign Operation 
(This appendix is an integral part of Ind AS 109) 
Background 
 
1 Many reporting entities have investments in foreign operations (as defined in Ind 
AS 21 pa

In [32]:
indas_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 10,
        "filter": {
            "source_file": "INDAS109.pdf"
        }
    }
)

docs = indas_retriever.invoke(
    "What is the objective of Ind AS 109?"
)

for rank, doc in enumerate(docs, start=1):

    print("=" * 80)

    print("Rank:", rank)

    print(
        "Page:",
        doc.metadata.get("page_number")
    )

    print(
        "Chunk:",
        doc.metadata.get("chunk_id")
    )

    print()

    print(doc.page_content[:400])

Rank: 1
Page: 188
Chunk: 621

433 
 
Appendix E 
References to matters contained in other Indian 
Accounting Standards 
This appendix is an integral part of the Ind AS. 
 
This appendix lists the appendices which are part of other Indian Accounting 
Standards and make reference to Ind AS 109,Financial Instruments. 
1. Appendix A,Rights to Interests arising from Decommissioning,Restoration 
and Environmental Rehabilitation cont
Rank: 2
Page: 173
Chunk: 580

418 
 
currency risks that qualify as a hedged risk in the hedge of a net investment in a 
foreign operation. 
 
5 Ind AS 109 allows an entity to designate either a de rivative or a non -derivative 
financial instrument (or a combination of derivative and non -derivative financial 
instruments) as hedging instruments for foreign currency risk. This Appendix 
provides guidance on where, within a gr
Rank: 3
Page: 172
Chunk: 577

417 
 
Appendix C 
Hedges of a Net Investment in a Foreign Operation 
(This appendix is an integral part of 

In [33]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "fetch_k": 30,
        "lambda_mult": 0.7
    }
)

docs = mmr_retriever.invoke(
    "What is the objective of Ind AS 109?"
)

for rank, doc in enumerate(docs, start=1):

    print("=" * 80)
    print("Rank:", rank)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page_number"))
    print("Chunk:", doc.metadata.get("chunk_id"))
    print()
    print(doc.page_content[:400])

Rank: 1
Source: INDAS109.pdf
Page: 188
Chunk: 621

433 
 
Appendix E 
References to matters contained in other Indian 
Accounting Standards 
This appendix is an integral part of the Ind AS. 
 
This appendix lists the appendices which are part of other Indian Accounting 
Standards and make reference to Ind AS 109,Financial Instruments. 
1. Appendix A,Rights to Interests arising from Decommissioning,Restoration 
and Environmental Rehabilitation cont
Rank: 2
Source: INDAS109.pdf
Page: 173
Chunk: 580

418 
 
currency risks that qualify as a hedged risk in the hedge of a net investment in a 
foreign operation. 
 
5 Ind AS 109 allows an entity to designate either a de rivative or a non -derivative 
financial instrument (or a combination of derivative and non -derivative financial 
instruments) as hedging instruments for foreign currency risk. This Appendix 
provides guidance on where, within a gr
Rank: 3
Source: INDAS109.pdf
Page: 31
Chunk: 89

contingent consideration recognised by an acqui

In [34]:
mmr_indas_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "fetch_k": 30,
        "lambda_mult": 0.7,
        "filter": {
            "source_file": "INDAS109.pdf"
        }
    }
)

docs = mmr_indas_retriever.invoke(
    "What is the objective of Ind AS 109?"
)

In [35]:
docs

[Document(id='44023ecf7a9d611cd8b763d1a45819847b3daf67460032d1cc63b0851ff1c696', metadata={'chunk_id': 621, 'creationdate': '2015-02-20T10:47:32+05:30', 'creator': 'Microsoft® Word 2013', 'chunk_length': 721, 'total_pages': 189, 'moddate': '2015-02-20T10:47:32+05:30', 'producer': 'Microsoft® Word 2013', 'page': 187, 'page_number': 188, 'author': 'nidhi1', 'page_label': '188', 'source': 'D:\\AI Projects\\Credit-Risk-Knowledge-Assistant\\documents\\INDAS109.pdf', 'source_file': 'INDAS109.pdf'}, page_content='433 \n \nAppendix E \nReferences to matters contained in other Indian \nAccounting Standards \nThis appendix is an integral part of the Ind AS. \n \nThis appendix lists the appendices which are part of other Indian Accounting \nStandards and make reference to Ind AS 109,Financial Instruments. \n1. Appendix A,Rights to Interests arising from Decommissioning,Restoration \nand Environmental Rehabilitation contained in Ind AS 37 , Provisions, \nContingent Liabilities and Contingent Asset

In [36]:
question = (
    "What objective is stated in Chapter 1 "
    "of Ind AS 109 Financial Instruments?"
)

docs = similarity_retriever.invoke(question)

for rank, doc in enumerate(docs, start=1):

    print("=" * 80)
    print("Rank:", rank)
    print("Source:", doc.metadata.get("source_file"))
    print("Page:", doc.metadata.get("page_number"))
    print("Chunk:", doc.metadata.get("chunk_id"))
    print()
    print(doc.page_content[:400])

Rank: 1
Source: INDAS109.pdf
Page: 1
Chunk: 0

246 
 
 
Indian Accounting Standard (Ind AS) 109 
Financial Instruments 
 
(The Indian Accounting Standard includes paragraphs set in bold type and plain 
type, which have equal authority. Paragraphs in bold type indicate the main 
principles.) 
 
 
Chapter 1 Objective 
 
1.1 The objective of this Standard is to establish principles for the 
financial reporting of financial assets and financial li
Rank: 2
Source: INDAS109.pdf
Page: 188
Chunk: 621

433 
 
Appendix E 
References to matters contained in other Indian 
Accounting Standards 
This appendix is an integral part of the Ind AS. 
 
This appendix lists the appendices which are part of other Indian Accounting 
Standards and make reference to Ind AS 109,Financial Instruments. 
1. Appendix A,Rights to Interests arising from Decommissioning,Restoration 
and Environmental Rehabilitation cont
Rank: 3
Source: INDAS109.pdf
Page: 128
Chunk: 413

B5.7.2 An entity applies Ind AS  21 to financial 